# Part 2: Fine-tuning InceptionV3 on Medical MNIST

This notebook fine-tunes ImageNet **InceptionV3** on **Medical MNIST**, the 6-class set of 64×64 images at [andrewmvd/medical-mnist](https://www.kaggle.com/datasets/andrewmvd/medical-mnist/data). The files used here are the public MedNIST release of that collection (same classes and the same 58,954 images). It was gathered by Dr. Bradley J. Erickson (Mayo Clinic) from TCIA, the RSNA Bone Age Challenge, and the NIH Chest X-ray dataset, and is shared under CC BY-SA 4.0.

The images are resized from 64×64 to 128×128. InceptionV3 needs at least 75×75, and with `include_top=False` the ImageNet weights still load at this size. Training has two stages. First the convolutional base stays frozen and only a new classification head is trained. Then the last Inception blocks are unfrozen and trained at a small learning rate, with batch-norm left frozen so the ImageNet statistics stay intact.

An 80/20 split (seed 42) uses 47,164 training images and 11,790 validation images. After fine-tuning, **validation accuracy is 99.97%**.


## Setup

In [ ]:
import json
import tarfile
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras

ROOT = Path.cwd()
DATA_DIR = ROOT / "data" / "MedNIST"
WEIGHTS = ROOT / "models" / "inception_v3_weights_tf_dim_ordering_tf_kernels_notop.h5"
OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

IMG_SIZE = 128
BATCH_SIZE = 64
SEED = 42
VAL_SPLIT = 0.2
HEAD_EPOCHS = 4
FINE_EPOCHS = 3
HEAD_LR = 1e-3
FINE_LR = 1e-5
UNFREEZE_AFTER = "mixed8"

keras.utils.set_random_seed(SEED)
print("TensorFlow", tf.__version__)

## Data

Six folders, one class each: AbdomenCT (10,000), BreastMRI (8,954), CXR (10,000), ChestCT (10,000), Hand (10,000), HeadCT (10,000). The download is the MedNIST archive from the MONAI project, which avoids needing a Kaggle API token in Colab. If `data/MedNIST` is already present, the download is skipped.

In [ ]:
MEDNIST_URL = (
    "https://github.com/Project-MONAI/MONAI-extra-test-data/releases/download/0.8.1/MedNIST.tar.gz"
)

if not DATA_DIR.is_dir():
    archive = ROOT / "data" / "MedNIST.tar.gz"
    archive.parent.mkdir(parents=True, exist_ok=True)
    print("Downloading Medical MNIST...")
    urllib.request.urlretrieve(MEDNIST_URL, archive)
    with tarfile.open(archive) as tar:
        tar.extractall(archive.parent)
    print("Extracted to", DATA_DIR)
else:
    print("Using existing", DATA_DIR)

common = dict(
    validation_split=VAL_SPLIT,
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int",
    color_mode="rgb",
)
train_ds = keras.utils.image_dataset_from_directory(DATA_DIR, subset="training", **common)
val_ds = keras.utils.image_dataset_from_directory(DATA_DIR, subset="validation", **common)
class_names = list(train_ds.class_names)
train_ds = train_ds.shuffle(512, seed=SEED).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)
print("Classes:", class_names)

In [ ]:
fig, axes = plt.subplots(1, len(class_names), figsize=(12, 2.4))
for ax, name in zip(axes, class_names):
    image_path = next(
        p for p in sorted((DATA_DIR / name).iterdir())
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )
    ax.imshow(plt.imread(image_path), cmap="gray")
    ax.set_title(name, fontsize=9)
    ax.axis("off")
fig.tight_layout()
plt.show()

## Model

InceptionV3 is loaded with ImageNet weights and without its original 1,000-way head. A global-average pool, dropout, and a 6-way softmax replace that head. Pixel values go through `inception_v3.preprocess_input`, which expects the 0–255 range produced by `image_dataset_from_directory`. A small random flip and rotation are applied only while training. The base is called with `training=False` so its batch-norm layers stay in inference mode during both stages.

In [ ]:
weights = str(WEIGHTS) if WEIGHTS.is_file() else "imagenet"
base = keras.applications.InceptionV3(
    include_top=False,
    weights=weights,
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
)
base.trainable = False

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = keras.layers.RandomFlip("horizontal")(inputs)
x = keras.layers.RandomRotation(0.05)(x)
x = keras.applications.inception_v3.preprocess_input(x)
x = base(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.2)(x)
outputs = keras.layers.Dense(len(class_names), activation="softmax")(x)
model = keras.Model(inputs, outputs)

def compile_model(learning_rate):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

compile_model(HEAD_LR)
model.summary()

## Stage 1 — train the new head

The convolutional base stays frozen, so only the new dense layer (12,294 parameters) is updated. Early stopping keeps the epoch with the best validation accuracy.

In [ ]:
early = keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=2, restore_best_weights=True
)
head_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=HEAD_EPOCHS,
    callbacks=[early],
    shuffle=False,
)
print("Frozen-base best val accuracy:", max(head_history.history["val_accuracy"]))

## Stage 2 — fine-tune the top Inception blocks

Layers after the `mixed8` block (the mixed9 and mixed10 modules) become trainable. Batch-norm layers stay frozen. The learning rate drops from `1e-3` to `1e-5` so the pretrained filters move only a little.

In [ ]:
cut_at = base.layers.index(base.get_layer(UNFREEZE_AFTER))
base.trainable = True
for layer in base.layers[: cut_at + 1]:
    layer.trainable = False
for layer in base.layers[cut_at + 1 :]:
    layer.trainable = not isinstance(layer, keras.layers.BatchNormalization)

compile_model(FINE_LR)
fine_early = keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=2, restore_best_weights=True
)
fine_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_EPOCHS,
    callbacks=[fine_early],
    shuffle=False,
)
print("Fine-tune best val accuracy:", max(fine_history.history["val_accuracy"]))

## Validation accuracy

In [ ]:
loss, accuracy = model.evaluate(val_ds, verbose=0)
print(f"Validation loss: {loss:.4f}")
print(f"Validation accuracy: {accuracy * 100:.2f}%")

matrix = np.zeros((len(class_names), len(class_names)), dtype=np.int64)
for images, labels in val_ds:
    pred = np.argmax(model.predict_on_batch(images), axis=1)
    for true, guess in zip(labels.numpy(), pred):
        matrix[int(true), int(guess)] += 1

recall = matrix.diagonal() / matrix.sum(axis=1).clip(min=1)
for name, value in zip(class_names, recall):
    print(f"{name:12s} recall {value * 100:.2f}%")

acc = head_history.history["accuracy"] + fine_history.history["accuracy"]
val_acc = head_history.history["val_accuracy"] + fine_history.history["val_accuracy"]
train_loss = head_history.history["loss"] + fine_history.history["loss"]
val_loss = head_history.history["val_loss"] + fine_history.history["val_loss"]
split = len(head_history.history["accuracy"])
epochs = np.arange(1, len(acc) + 1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(epochs, acc, "o-", color="#7a1f2b", label="Train")
axes[0].plot(epochs, val_acc, "o-", color="#1f4b99", label="Validation")
axes[0].axvline(split + 0.5, color="#888888", ls="--", lw=1)
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend(frameon=False)
axes[1].plot(epochs, train_loss, "o-", color="#7a1f2b", label="Train")
axes[1].plot(epochs, val_loss, "o-", color="#1f4b99", label="Validation")
axes[1].axvline(split + 0.5, color="#888888", ls="--", lw=1)
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend(frameon=False)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "part2_training_curves.png", dpi=140)
plt.show()

fig, ax = plt.subplots(figsize=(7.2, 6))
im = ax.imshow(matrix, cmap="Reds")
fig.colorbar(im, ax=ax, fraction=0.046)
ax.set_xticks(range(len(class_names)), class_names, rotation=40, ha="right")
ax.set_yticks(range(len(class_names)), class_names)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Validation confusion matrix — {accuracy * 100:.2f}% accuracy")
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        color = "white" if matrix[i, j] > matrix.max() / 2 else "black"
        ax.text(j, i, str(matrix[i, j]), ha="center", va="center", color=color, fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "part2_confusion_matrix.png", dpi=140)
plt.show()

results = {
    "model": "InceptionV3",
    "dataset": "Medical MNIST (MedNIST)",
    "validation_accuracy": float(accuracy),
    "validation_loss": float(loss),
    "per_class_recall": {n: float(v) for n, v in zip(class_names, recall)},
}
(OUTPUT_DIR / "part2_results.json").write_text(json.dumps(results, indent=2))
results

## Writeup

InceptionV3, pretrained on ImageNet, was fine-tuned on Medical MNIST. The set has 58,954 images at 64×64 in six classes: abdomen CT, breast MRI, chest X-ray, chest CT, hand X-ray, and head CT. Each image was resized to 128×128 and split 80/20. The new classification head was trained for four epochs with the convolutional base frozen, which reached 99.80% validation accuracy. The last two Inception blocks were then unfrozen and trained for three epochs at a learning rate of 1e-5.

The best validation accuracy is **99.97%** (11,787 of 11,790 images, loss 0.0007). Breast MRI, chest X-ray, hand, and head CT were classified perfectly. The three mistakes are all between CT classes: two abdomen CT slices called head CT, and one chest CT slice called abdomen CT. On the training curves, the dashed line is where fine-tuning starts. Validation accuracy dipped from 99.91% to 99.79% on the second fine-tuning epoch, then rose to 99.97% on the third.

![Training curves](output/part2_training_curves.png)

![Validation confusion matrix](output/part2_confusion_matrix.png)
